# Import Dataset FashionMNIST

1. Split training data (60.0000) and testing data (10.000)
2. Split data and target
3. Input data Tensor for training_data $ \mathbf{T} \in \mathbb{R}^{60000 \times 28 \times 28} $

In [2]:
import torch
# from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [3]:
training_data = datasets.FashionMNIST(
    root="data",         # Directory where data will be stored
    train=True,          # Load training dataset (60,000 samples)
    download=True,       # Download if not present locally
    transform=ToTensor() # Convert PIL Image or numpy array to a PyTorch Tensor
)

test_data = datasets.FashionMNIST(
    root="data",
    train=False,         # Load testing dataset (10,000 samples)
    download=True,
    transform=ToTensor()
)

In [4]:
print(training_data.data.shape)
print(training_data.targets.shape)

torch.Size([60000, 28, 28])
torch.Size([60000])


# Flatten Tensor T to matrix X

1. $\mathbf{X} \in \mathbb{R}^{60000 \times 784} $
3. Normalize matrix $ \mathbf{X} \in [0,1] $ by divide it 255 (min = 0px, max= 255px)

In [5]:
import torch.nn as nn
flatten = nn.Flatten()
X = flatten(training_data.data).float() / 255.0

print(X.shape)

torch.Size([60000, 784])


# Convert target (Y) to one-hot encoded matrix

$ \mathbf{Y} \in \{0,1\}^{60000 \times k} $, $ k = 10$

In [6]:
import torch.nn.functional as F

Y = F.one_hot(training_data.targets, num_classes=10).float()

Y.shape

torch.Size([60000, 10])

# Generate random vector w (weight) and b (bias)
1. Because target $ \mathbf{Y} $ has $ k = 10$, we generate  $ \mathbf{W} \in \mathbb{R}^{784 \times 10} $, $ \mathbf{W}_{i,j} \sim U[0,1] $ and $ \mathbf{b} \in \mathbb{R}^{10}$ and $ \mathbf{b}_{i} \sim U[0,1]$
2. Let $ F : \mathbb{R}^{60000 \times 784} \to \mathbb{R}^{60000 \times 10} $ and $ \mathbf{Z} = F(\mathbf{X}) $ then $ \mathbf{Z} = \mathbf{X} \mathbf{W} + \mathbf{b} $.  $\mathbf{b} $ is broadcast, the dimension becomes $ \mathbb{R}^{1 \times 10} $
3. Normalise the $ \mathbf{Z}_{i} $ value with $ \mathbf{Z}_{i} = \mathbf{Z}_{i} - max(Z_{i}) $ where i is row of matrix $ \mathbf{Z} $

In [7]:
W = torch.rand(X.shape[1],Y.shape[1])
b = torch.rand(Y.shape[1])

# linear function
Z = X @ W + b
Z = Z - torch.max(Z, dim=-1, keepdim=True).values
Z.shape

torch.Size([60000, 10])

# Activation Function for Multinomial Classification
Let $ \sigma : \mathbb{R}^{60000 \times 10} \to (0,1)^{60000 \times 10} $, we can define $ \mathbf{z}_{i} = [z_{i,1}, z_{i,2}, z_{i,3}, \dots, z_{i,k}] $ where $ i = 1,2...,60000 $ and $ k = 10 $ we can formulate our activation function  

$ \sigma (\mathbf{Z})_i =  \frac{e^{\mathbf{z}_{i}}}{\sum_{j=1}^{k} e^{z_{i,j}}} $ where $ \sum_{j=1}^{k} \sigma (\mathbf{Z})_{i,j} = 1$  

$ \sum_{j=1}^{k} e^{z_{i,j}} $ firstly is $ \mathbb{R}^{60000}$ and it broadcasts and changes dimension $ \mathbb{R}^{60000 \times 10} $

In [8]:
S = torch.exp(Z) / torch.sum(torch.exp(Z), dim=1, keepdim=True) # keepdim is keeping the dimension from 60000 (1D) vector to 60000x1 (2D).
S.shape

torch.Size([60000, 10])

# Loss Function using NLL

Let $ Loss : \mathbb{R}^{60000 \times 10} \to [0,\infty)^{60000}$  we can define loss function  
$ Loss(\mathbf{Z})_{i} = - \sum_{j=1}^{k} y_{i,j} log(\sigma(\mathbf{Z})_{i,j})$ where $ i = 1,2,..., 60000 $ and $ k = 10 $  
or  

$ Loss(\mathbf{Z})_{i} = -log(\sigma(\mathbf{Z})_{i}) \mathbf{Y}_{i}^\intercal = -\langle log(\sigma(\mathbf{Z})_{i}),  \mathbf{Y}_{i} \rangle$ where $ \sigma(\mathbf{Z})_{i} \in (0,1)^{1 \times 10} $, $ \sum_{j=1}^{k} \sigma (\mathbf{Z})_{i,j} = 1$, and $ \mathbf{Y}_{i} \in \{0,1\}^{1 \times 10}, \sum_{j=1}^{k} y_{i,j} = 1 $

In [9]:
Loss = torch.sum(-torch.log(S) * Y, dim=1)
Loss.shape

torch.Size([60000])

# Sum of functions
We want to minimize loss a function $ Loss(\mathbf{Z}) $, $ f : \mathbb{R}^{60000 \times 10} \to [0,\infty)$ which writes as $ f(\mathbf{Z}) = \frac{1}{n} \sum_{i=1}^{n}f_{i}(\mathbf{Z}) $ where $ Loss(x)_{i} = f_{i}(x)$

In [10]:
sum_of_func = torch.mean(Loss)

sum_of_func

tensor(4.8891)

# Derivation

We flatten the dimension of $ \mathbf{Z} $ into vector $ \mathbf{z} $ in order to easily calculate for creating Jacobian matrix.  Suppose the dimensions become $ \sigma : \mathbb{R}^{m} \to (0,1)^{m}$.  We can define $ p_i $ as the softmax activation function for all elements $ i $.   Based on function $ \sigma(\mathbf{z})_i $ we can write the variable  $ p_i = \frac{e^{z_i}}{\sum_{k} e^{z_{k}}} $ and vector $\mathbf{p} $ contains variable $p_i$

## First-order derivative
We need to find the first-order derivative for the function of $ \sigma{} $, and therefore $\nabla \sigma : \mathbf{R}^{m} \to (0,1)^{m \times m}$.  Based on $ p_i $ variable, there are two conditions for its derivation; derivative of $ p_i $ with respect to $ z_i $, and derivative of $ p_i $ with respect other variable $ z_i $.  Assume the other variable is $ z_j $ we can put our differentiation formula $ \nabla \sigma(\mathbf{z})_{i} = \begin{cases} \frac{\partial p_i}{\partial z_i} = \frac{e^{z_i}(\sum_{k}e^{z_k} - e^{z_i})}{(\sum_{k}e^{z_k})^2} = p_i(1-p_i), i=j \\ \frac{\partial p_i}{\partial z_j} = \frac{-e^{z_i}e^{z_j}}{(\sum_{k} e^{z_{k}})^2} = -p_ip_j, i \neq j \end{cases}$  
We can distribute the formula $ p_i(1-p_i) =  p_i - p_i^2 $.  In the first term, we can define the matrix diagonal $ diag(\mathbf{p}) $ and for the last term, we can combine with the formula when $ i \neq j $ using outer product $ \mathbf{-pp\top} $.  So the result is $ \nabla \sigma(\mathbf{z}) = diag(\mathbf{p}) - \mathbf{pp\top}$  

Continue to find the derivative of $ Loss(\mathbf{z})_i $ changing flatten dimensions to $\nabla Loss : \mathbf{R}^{m} \to [0,\infty)^{m}$, and $ \mathbf{Y} \in \{0,1\}^{60000 \times 10}$ also flatten into vector $ \mathbf{y} \in \{0,1\}^{m}$we rewrite the formula of $ Loss(\mathbf{z})_{i} = - \sum_{j=1}^{k} y_j log(\sigma(\mathbf{z})_j) $ and we can find the derivative:
$$ 
\begin{aligned}
\nabla Loss(\mathbf{z})_{i} &= - \sum_{j=1}^{k} y_j \frac{1}{\sigma(\mathbf{z})_j}\nabla \sigma(\mathbf{z})_{j} \\
&= -\left(\underbrace{y_i \frac{1}{\sigma(\mathbf{z})_i}\nabla \sigma(\mathbf{z})_{i}}_{i = j} + \underbrace{\sum_{i \neq j}^{k} y_j \frac{1}{\sigma(\mathbf{z})_j}\nabla \sigma(\mathbf{z})_{j}}_{i \neq j} \right) \\
&= -\left(y_i \frac{1}{p_i}\frac{\partial p_i}{\partial z_i} + \sum_{i \neq j}^{k} y_j \frac{1}{p_j}\frac{\partial p_i}{\partial z_j}\right) \\
&= -\left(y_i \frac{1}{p_i} p_i (1-p_i) + \sum_{i \neq j}^{k} -y_j \frac{1}{p_j} p_i p_j \right) \\
&= -\left(y_i (1 - p_i) + \sum_{i \neq j}^{k} -y_j p_i \right) \\
&= -y_i + p_i \underbrace{\left(y_i + \sum_{i \neq j}^k y_j \right)}_{ = 1} \\
\nabla Loss(\mathbf{z}) &= \mathbf{p} - \mathbf{y}
\end{aligned}
$$

After we find $ \nabla Loss(\mathbf{z}) $ we can apply to sum of functions, when it is flatten $ f : \mathbb{R}^{m} \to [0,\infty) $ and let $ \nabla f : \mathbb{R}^{m} \to [0,\infty)^{m} $, we can write $ \nabla f(\mathbf{z}) = \frac{1}{n} \sum_{i=1}^{n} \nabla f_i(\mathbf{z})$  

We have two variables that should be differentiate $\mathbf{W} $ and $\mathbf{b} $ so we can write  

$ \frac{\partial f}{\partial \mathbf{w}} = \frac{\partial f}{\partial \mathbf{z}} \frac{\partial \mathbf{z}}{\partial \mathbf{w}} =  \frac{1}{n} \sum_{i=1}^{n} \sum_{j=1}^{m} x_{i,j}\nabla f_j(\mathbf{z}) = \frac{1}{n} \sum_{n} \mathbf{X}\top  \left(\mathbf{p} - \mathbf{y} \right) $   
$ \frac{\partial f}{\partial \mathbf{b}} = \frac{\partial f}{\partial \mathbf{z}} \frac{\partial \mathbf{z}}{\partial \mathbf{b}} =  \frac{1}{n} \sum_{i=1} \nabla f_i(\mathbf{z}) = \frac{1}{n} \sum_{n} \mathbf{p} - \mathbf{y} $   


In [11]:
def derive_loss(S):
    return S - Y

def derive_sum_of_function(d_L):
    return d_L.mean(dim=0)

dLoss = derive_loss(S)
dZ_dW = X
df_dW = derive_sum_of_function(dZ_dW.T @ dLoss)
df_dB = derive_sum_of_function(dLoss)

print(df_dW.shape)
print(df_dB.shape)


torch.Size([10])
torch.Size([10])


## Second-order derivative

Let $ \nabla f : \mathbb{R}^{m} \to \mathbb{R}^{m} $, then we have $ \nabla^2 f : \mathbb{R}^{m} \to \mathbb{R}^{m \times m} $.  We can find its Hessian matrix :  
$$ \frac{\partial^2 f}{\partial w^2} = \frac{\partial \left(\frac{1}{n} \sum_{i=1} \sum_{j=1} x_{i,j}(p_j-y_j) \right)}{\partial z} \frac{\partial z}{\partial w} = \frac{1}{n} \sum_{n} \mathbf{X}\top \nabla \sigma(\mathbf{z}) \mathbf{X} $$
$$ \frac{\partial^2 f}{\partial w \partial b} = \frac{\partial \left(\frac{1}{n} \sum_{i=1} (p_i-y_i) \right)}{\partial z} \frac{\partial z}{\partial w} = \frac{1}{n} \sum_{n} \nabla \sigma(\mathbf{z}) \mathbf{X} $$
$$ \frac{\partial^2 f}{\partial b \partial w} = \frac{\partial \left(\frac{1}{n} \sum_{i=1} \sum_{j=1} x_{i,j}(p_j-y_j) \right)}{\partial z} \frac{\partial z}{\partial b} = \frac{1}{n} \sum_{n} \mathbf{X}\top \nabla \sigma(\mathbf{z}) $$
$$ \frac{\partial^2 f}{\partial b^2} = \frac{\partial \left(\frac{1}{n} \sum_{i=1} \sum_{j=1} (p_j-y_j) \right)}{\partial z} \frac{\partial z}{\partial b} = \frac{1}{n} \sum_{n} \nabla \sigma(\mathbf{z}) $$  
$$\nabla^2 f = \frac{1}{n}\begin{bmatrix} \sum_{n} \mathbf{X}\top \nabla \sigma(\mathbf{z}) \mathbf{X} & \sum_{n} \nabla \sigma(\mathbf{z}) \mathbf{X} \\ \sum_{n} \mathbf{X}\top \nabla \sigma(\mathbf{z}) &  \sum_{n} \nabla \sigma(\mathbf{z}) \end{bmatrix} $$  

However, we cannot use all values in Hessian matrix because of 


In [12]:
diag_S = torch.diag_embed(S)
outer_S = torch.einsum('bi,bj->bij', S, S)
Hessian = diag_S - outer_S

Hessian

tensor([[[ 1.0428e-02, -3.8454e-04, -1.2872e-03,  ..., -5.7337e-04,
          -5.6641e-08, -3.2558e-06],
         [-3.8454e-04,  3.5156e-02, -4.4566e-03,  ..., -1.9851e-03,
          -1.9610e-07, -1.1272e-05],
         [-1.2872e-03, -4.4566e-03,  1.0722e-01,  ..., -6.6451e-03,
          -6.5644e-07, -3.7734e-05],
         ...,
         [-5.7337e-04, -1.9851e-03, -6.6451e-03,  ...,  5.1445e-02,
          -2.9240e-07, -1.6808e-05],
         [-5.6641e-08, -1.9610e-07, -6.5644e-07,  ..., -2.9240e-07,
           5.3745e-06, -1.6604e-09],
         [-3.2558e-06, -1.1272e-05, -3.7734e-05,  ..., -1.6808e-05,
          -1.6604e-09,  3.0884e-04]],

        [[ 3.2296e-04, -3.7317e-07, -1.2460e-04,  ..., -6.5177e-05,
          -1.4302e-12, -1.1669e-06],
         [-3.7317e-07,  1.1537e-03, -4.4550e-04,  ..., -2.3303e-04,
          -5.1136e-12, -4.1721e-06],
         [-1.2460e-04, -4.4550e-04,  2.3693e-01,  ..., -7.7809e-02,
          -1.7075e-09, -1.3931e-03],
         ...,
         [-6.5177e-05, -2

# Convexity
1. To determine whether the function is convex, we can define convexity : let $ F: \mathbb{R}^{m \times n} \to \mathbb{R} \cup \{+\infty\} $ is convex if $ \forall x,y \in \mathbb{R}^{m \times n}, \forall t \in (0,1), F(t$

In [13]:
# Tensor batch (misal: 2 batch, masing-masing punya vektor ukuran 3)
v_batch = torch.tensor([[1, 2, 3], 
                        [4, 5, 6]])

# Membuat batch matriks diagonal (hasilnya 3D: 2x3x3)
batch_diag = torch.diag_embed(v_batch)

batch_diag

tensor([[[1, 0, 0],
         [0, 2, 0],
         [0, 0, 3]],

        [[4, 0, 0],
         [0, 5, 0],
         [0, 0, 6]]])